In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
import xml.etree.ElementTree as ET
from datetime import datetime

# ── Configuration ──────────────────────────────────────────────────────
CATALOG         = "bfsi"
SCHEMA          = "bronze_layer"
SOURCE_DIR      = "/Volumes/bfsi/landing/neft_rtgs/"  # Auto Loader directory
TABLE_NAME      = f"{CATALOG}.{SCHEMA}.neft_rtgs_transactions"
CHECKPOINT_PATH = f"/Volumes/bfsi/landing/neft_rtgs/_checkpoint/{SCHEMA}_neft_rtgs_transactions"
SOURCE_SYSTEM   = "NEFT_RTGS"
BATCH_ID        = dbutils.widgets.get("batch_id") if "batch_id" in [w.name for w in dbutils.widgets.getAll()] else f"BATCH_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

In [0]:
# ── FIX: Build per-file header map from ALL XML files ────────────────
# Previously, header values (msg_id, total_settlement_amt) were extracted
# from a SINGLE hardcoded file and applied as static literals to ALL records.
# This caused the BALANCE check to fail when multiple XML files were present,
# because each file has its own GrpHdr with different values.
#
# Now we scan every .xml file in SOURCE_DIR and build a lookup dict:
#   filename -> (msg_id, total_settlement_amt)

xml_files = [f.path for f in dbutils.fs.ls(SOURCE_DIR) if f.name.endswith(".xml")]

_neft_header_map = {}  # filename -> (msg_id, total_settlement_amt)

for xml_path in xml_files:
    xml_content = dbutils.fs.head(xml_path, 10_000_000)  # up to 10 MB
    tree = ET.fromstring(xml_content)
    grp_hdr = tree.find(".//GrpHdr")
    file_msg_id = grp_hdr.find("MsgId").text
    total_amt_element = grp_hdr.find("TtlIntrBkSttlmAmt")
    file_total_amt = float(total_amt_element.text)
    file_ccy = total_amt_element.get("Ccy", "INR")

    fname = xml_path.rsplit("/", 1)[-1]
    _neft_header_map[fname] = (file_msg_id, file_total_amt)

    nb_of_txs = grp_hdr.find("NbOfTxs").text
    print(f"File: {fname}")
    print(f"  Message ID:              {file_msg_id}")
    print(f"  Number of Transactions:  {nb_of_txs}")
    print(f"  Total Settlement Amount: {file_ccy} {file_total_amt:,.2f}")
    print()

print(f"Loaded headers for {len(_neft_header_map)} XML file(s)")

In [0]:
# ── UDFs: resolve correct header values per record's source file ─────

@F.udf(StringType())
def _get_neft_msg_id(file_path):
    """Return the msg_id for the XML file that produced this record."""
    if file_path:
        fname = file_path.rsplit("/", 1)[-1]
        entry = _neft_header_map.get(fname)
        return entry[0] if entry else None
    return None

@F.udf(DoubleType())
def _get_neft_total_amt(file_path):
    """Return the total_settlement_amt for the XML file that produced this record."""
    if file_path:
        fname = file_path.rsplit("/", 1)[-1]
        entry = _neft_header_map.get(fname)
        return entry[1] if entry else None
    return None

print("UDFs registered: _get_neft_msg_id, _get_neft_total_amt")

In [0]:
# ── Schema for CdtTrfTxInf elements ─────────────────────────────────
neft_xml_schema = StructType([
    StructField("InstrId", StringType(), True),
    StructField("Amt", StructType([
        StructField("_VALUE", DoubleType(), True),
        StructField("_Ccy", StringType(), True),
    ]), True),
    StructField("Dbtr", StructType([
        StructField("Nm", StringType(), True),
    ]), True),
    StructField("Cdtr", StructType([
        StructField("Nm", StringType(), True),
    ]), True),
    StructField("Purp", StructType([
        StructField("Cd", StringType(), True),
    ]), True),
    StructField("RmtInf", StructType([
        StructField("Ustrd", StringType(), True),
    ]), True),
])

# Auto Loader: incremental ingestion of XML files
df_neft_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "xml")
    .option("cloudFiles.includeExistingFiles", "true")
    .option("rowTag", "CdtTrfTxInf")
    .schema(neft_xml_schema)
    .load(SOURCE_DIR)
)

print("Auto Loader stream configured for NEFT/RTGS XML transactions")

In [0]:
# ── Flatten nested XML structs and add metadata columns ──────────────
# FIX: msg_id and _total_settlement_amt are now resolved PER-FILE via UDFs
# instead of being static literals from a single hardcoded file.

df_neft_bronze = (
    df_neft_stream
    .select(
        F.col("InstrId").alias("instr_id"),
        F.col("Amt._VALUE").alias("amount"),
        F.col("Amt._Ccy").alias("currency"),
        F.col("Dbtr.Nm").alias("debtor_name"),
        F.col("Cdtr.Nm").alias("creditor_name"),
        F.col("Purp.Cd").alias("purpose_code"),
        F.col("RmtInf.Ustrd").alias("remittance_info"),
        # Preserve raw parsed XML struct as JSON for programmatic access
        F.to_json(
            F.struct("InstrId", "Amt", "Dbtr", "Cdtr", "Purp", "RmtInf")
        ).alias("_raw_xml_content"),
    )
    # FIX: Use UDFs to resolve per-file header values
    .withColumn("msg_id", _get_neft_msg_id(F.col("_metadata.file_path")))
    .withColumn("_source_system", F.lit(SOURCE_SYSTEM))
    .withColumn("_load_ts", F.current_timestamp())
    .withColumn("_file_name", F.col("_metadata.file_path"))
    .withColumn("_batch_id",
        F.concat(F.lit("BATCH_NEFT_"), _get_neft_msg_id(F.col("_metadata.file_path")))
    )
    .withColumn("_total_settlement_amt", _get_neft_total_amt(F.col("_metadata.file_path")))
)

# Write stream to Delta table in bronze_layer
query = (
    df_neft_bronze.writeStream
    .format("delta")
    .option("checkpointLocation", CHECKPOINT_PATH)
    .option("mergeSchema", "true")
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable(TABLE_NAME)
)

query.awaitTermination()
print(f"Auto Loader stream completed for {TABLE_NAME}")

In [0]:
# ── Verify data in bronze Delta table ────────────────────────────────
df_verify = spark.table(TABLE_NAME)
print(f"Total NEFT/RTGS transactions ingested: {df_verify.count()}")
display(df_verify.limit(5))